# Cuaderno 5 — Spline cúbico y datos reales (USGS Water Data) · Integrante B
**Fuentes:** USGS Water Data — https://waterdata.usgs.gov/nwis · SciPy `CubicSpline` — https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.CubicSpline.html · SciPy interpolación — https://docs.scipy.org/doc/scipy/reference/interpolate.html
**Ejercicio enlazado:** 15 (spline natural manual, compartido; se incorpora una sola vez). **Módulos:** `interpolacion.py`, `datos.py`, `metricas.py`.

## 1. Spline cúbico natural
En cada tramo $S_i$ es cúbico con $S,S',S''$ continuas. Con $M_i=S''(x_i)$, $h_i=x_{i+1}-x_i$, $\delta_i=(y_{i+1}-y_i)/h_i$:
$$h_{i-1}M_{i-1}+2(h_{i-1}+h_i)M_i+h_iM_{i+1}=6(\delta_i-\delta_{i-1}),\qquad M_0=M_n=0 .$$
**Natural** = curvatura nula en los extremos: la curva se aplana en los bordes, lo que puede aumentar el error cerca de ellos si el fenómeno tiene curvatura ahí.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from numpy.polynomial import polynomial as P
from scipy.interpolate import BarycentricInterpolator, lagrange as sp_lagrange, CubicSpline
from interpolacion import *
from datos import cargar, control_calidad, particion_comun, comparar, fraccion_regular, codigos_calificacion
import metricas
pd.options.display.float_format = '{:.5f}'.format

In [ ]:
xs, ys = np.array([0., 1, 2]), np.array([0., 1, 0])
M = spline_natural_momentos(xs, ys); print("Momentos (manual 0, -3, 0):", M)
print("S0(x) = a+b x+c x²+d x³, (a,b,c,d) =", tramo_spline(xs, ys, 0, M), "  (manual 0, 3/2, 0, -1/2)")
print("S0(0.5): manual", -0.5*0.125 + 1.5*0.5, "| propio", spline_natural_eval(xs, ys, [0.5])[0], "| SciPy", spline_natural_scipy(xs, ys, [0.5])[0])
print("S'' en extremos (SciPy):", CubicSpline(xs, ys, bc_type="natural")(xs, 2)[[0, -1]])

## 2. Base real: documentación y control de calidad
| Campo | Valor |
|---|---|
| Institución | USGS — National Water Information System |
| Estación / variable | USGS 01646500, Potomac River near Washington, DC (el encabezado del archivo entregado dice «UPPER DAMSITE»; la ficha oficial dice «LITTLE FALLS PUMP STA»: verificar) · caudal medio diario, parámetro 00060, estadístico 00003, ft³/s |
| URL / consulta | https://waterservices.usgs.gov/nwis/dv/?format=rdb&sites=01646500&parameterCd=00060&statCd=00003&startDT=2025-01-01&endDT=2025-12-31 *(si el servicio heredado ya no responde, registrar la consulta realmente usada)* |
| Fecha de acceso | *(la del día real de la descarga)* |
| Condiciones de uso | Datos públicos USGS; citar la fuente; los datos pueden ser provisionales |
| Original | `data/usgs_caudal_diario.rdb` (sin modificar) · Procesada: `data/usgs_procesado.csv` |
| Procesamiento | Se omiten comentarios y fila de formatos; conversión numérica; se eliminan duplicados; valores no numéricos (p. ej. *Ice*, *Eqp*) y ausentes se **excluyen sin rellenar** y se cuentan; recorte de 36 días consecutivos |

In [ ]:
df, es_real = cargar(n_recorte=36); full, _ = cargar(n_recorte=10**6, guardar=False)
print("BASE USGS (archivo local)" if es_real else "⚠ DATOS SINTÉTICOS")
pd.options.display.float_format = '{:.4f}'.format
display(pd.Series(control_calidad(full)).to_frame("serie completa")); display(pd.Series(control_calidad(df)).to_frame("recorte"))
display(df.head())

In [ ]:
sitio = full.attrs.get("site_no")
print("Estación en el archivo:", sitio, "| rango:", full["fecha"].min().date(), "→", full["fecha"].max().date(), "| días válidos:", len(full), "| excluidos no numéricos:", full.attrs.get("excluidos_no_numericos"))
if es_real and sitio != "01646500": print("⚠ La estación del archivo no coincide con la documentada (01646500): actualice la ficha o vuelva a descargar.")
if es_real:
    print("Códigos de calificación:", codigos_calificacion("data/usgs_caudal_diario.rdb").to_dict())
    fr = fraccion_regular(full["valor"]); print(f"Fracción de segundas diferencias casi constantes: {fr:.2f}", "⚠ SERIE SOSPECHOSAMENTE REGULAR (una serie real suele dar < 0.1)" if fr > 0.3 else "(coherente con datos con ruido)")
if es_real: print("Compare la mediana (", round(float(full['valor'].median()), 1), "ft³/s ) con la página de la estación: https://waterdata.usgs.gov/monitoring-location/01646500")

**Verificación de la descarga.** Antes de interpretar resultados confirme: estación 01646500, parámetro 00060, estadístico 00003, fechas pedidas, cuántos valores se excluyeron y sus códigos de calificación, y que el orden de magnitud sea coherente con la ficha oficial de la estación. Conserve el `.rdb` original intacto. **Si la celda anterior marca la serie como sospechosamente regular, los resultados NO deben presentarse como datos reales**: vuelva a descargar el archivo desde USGS.

## 3. Partición común
Extremos siempre como nodos; se reservan observaciones **interiores** (omisiones aisladas cada $p$ días y un hueco interior corto); no se rellena nada previamente; los **mismos** nodos y reservas para Lagrange, Newton y spline. Tiempo normalizado a $[0,1]$ para el condicionamiento.

In [ ]:
PASO, HUECO = 4, 2
nodos, val = particion_comun(len(df), PASO, HUECO); R = comparar(df, nodos, val, "ft³/s")
print("Nodos:", len(nodos), "| reservadas:", len(val), "| índices reservados:", val.tolist())
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(R["t"][nodos], R["y"][nodos], "ko", ms=4, label="nodos"); ax.plot(R["t_val"], R["y_val"], "rs", label="reservadas")
for k, v in R["curvas"].items(): ax.plot(R["t_curva"], v, lw=1, label=k)
ax.set_ylim(R["y"].min()-40, R["y"].max()+40); ax.set_xlabel("día"); ax.set_ylabel("Q [ft³/s]"); ax.legend(fontsize=7); plt.show()

## 4. Errores con signo y métricas
$e_i=\hat y_i-y_i$; $\mathrm{MAE}=\frac1m\sum|e_i|$; $\mathrm{RMSE}=\sqrt{\frac1m\sum e_i^2}$; $E_{\max}=\max|e_i|$, todas en ft³/s.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5)); w = 0.2
for j, (k, v) in enumerate(R["pred"].items()): ax.bar(np.arange(len(val)) + (j-1.5)*w, metricas.errores(v, R["y_val"]), w, label=k)
ax.axhline(0, color="k", lw=.6); ax.set_xticks(range(len(val))); ax.set_xticklabels([f"día {t:.0f}" for t in R["t_val"]], rotation=45, fontsize=7)
ax.set_ylabel("e = ŷ − y [ft³/s]"); ax.legend(fontsize=7); plt.show()
pd.options.display.float_format = '{:.3f}'.format
display(R["metricas"].rename(columns={"MAE": "MAE [ft³/s]", "RMSE": "RMSE [ft³/s]", "E_max": "E_máx [ft³/s]"}))

## 5. Reproducción de nodos y acuerdo entre implementaciones

In [ ]:
display(pd.Series({k: f"{v:.2e}" for k, v in R["reproduccion_nodos"].items()}).to_frame("residual en nodos [ft³/s]"))
display(pd.Series({k: f"{v:.2e}" for k, v in R["acuerdo"].items()}).to_frame("discrepancia [ft³/s]"))

**Qué aporta cada comprobación.** *Residual en nodos* ≈ 0: pasa por los datos (en Newton global es pequeño pero no nulo, por condicionamiento). *Lagrange–Newton* y *spline propio–SciPy*: validan la **implementación**; Lagrange–Newton no mide el error frente al río. *Métricas en reservadas*: discrepancia frente a mediciones con incertidumbre, no una cota del error físico.

## 6. Omisiones aisladas vs. hueco interior y sensibilidad

In [ ]:
filas = []
for p, h in [(3, 0), (4, 0), (4, 2), (4, 4), (6, 0)]:
    n_, v_ = particion_comun(len(df), p, h); r_ = comparar(df, n_, v_)
    for k, row in r_["metricas"].iterrows():
        if k != "Spline natural (SciPy)": filas.append({"paso": p, "hueco": h, "nodos": len(n_), "Método": k, "E_max": row.E_max})
display(pd.DataFrame(filas).pivot_table(index=["paso", "hueco", "nodos"], columns="Método", values="E_max"))
print("Extrapolación desactivada (NaN fuera de [x_min, x_max]):", spline_natural_eval(xs, ys, [-1, 3]))

## 7. Cierre e interpretación
*(Redactar con las cifras de la ejecución sobre la base verificada; `python reporte_resultados.py` genera las tablas.)* Puntos a discutir: (i) un polinomio global de grado alto con nodos equiespaciados puede oscilar entre nodos (Runge), mientras el spline cúbico es local; compare los E_máx obtenidos y cómo cambian con la partición; (ii) el spline no garantiza positividad ni monotonía; (iii) reconstruir datos interiores no demuestra capacidad de pronóstico; (iv) las discrepancias son frente a mediciones con incertidumbre, no una cota del error físico; (v) no se presupone un método ganador. **Límites:** un solo recorte, una sola estación.

**Registro de IA:** ver `registro_IA.md` y `history.md`. *(Completar con fecha, herramienta/modelo tal como se muestra, prompt exacto, qué se aceptó/modificó/descartó y la comprobación independiente; o declarar «Sin uso de IA».)*